# 📊 Phân Tích Dữ Liệu Sau Tiền Xử Lý — `clean_data_all.csv`

> **Mục tiêu**: Kiểm tra tính chất, phân bố, và chất lượng dữ liệu sau khi chạy preprocessing pipeline.
> Dùng cho **báo cáo / luận văn** về Air Quality Forecasting tại Việt Nam.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (16, 8),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight'
})

# Color palettes
AQ_COLORS = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6', '#1abc9c']
PROVINCE_COLORS = sns.color_palette('Set2', 12)

print('Libraries loaded OK ✅')

## 1️⃣ Load Dữ Liệu & Tổng Quan

In [ ]:
# Load data
DATA_PATH = 'data/clean_data_all.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp_local'])

print(f'📁 File: {DATA_PATH}')
print(f'📐 Shape: {df.shape[0]:,} dòng × {df.shape[1]} cột')
print(f'📅 Thời gian: {df["timestamp_local"].min()} → {df["timestamp_local"].max()}')
print(f'🏙️  Số trạm: {df["location"].nunique()}')
print(f'🏠 Số tỉnh: {df["province"].nunique()}')
print(f'\n📋 Danh sách cột ({len(df.columns)}):')
for i, col in enumerate(df.columns):
    dtype = df[col].dtype
    print(f'   {i+1:>2}. {col:<25} ({dtype})')

In [ ]:
# Phân bố trạm
print('\n🏙️  Phân bố dữ liệu theo trạm:')
station_counts = df['location'].value_counts()
for loc, count in station_counts.items():
    province = df[df['location'] == loc]['province'].iloc[0]
    pct = count / len(df) * 100
    print(f'   {province:<12} | {loc:<25} | {count:>7,} dòng ({pct:.1f}%)')

## 2️⃣ Missing Values — Kiểm Tra Dữ Liệu Thiếu

In [ ]:
# Tổng missing
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, '%': missing_pct}).sort_values('Missing', ascending=False)
missing_df = missing_df[missing_df['Missing'] > 0]

if len(missing_df) == 0:
    print('✅ KHÔNG CÒN DỮ LIỆU THIẾU — Pipeline imputation hoạt động tốt!')
else:
    print(f'⚠️ Còn {len(missing_df)} cột có dữ liệu thiếu:')
    display(missing_df)

# Missing per station
print('\n📊 Missing values theo từng trạm:')
for loc in df['location'].unique():
    loc_df = df[df['location'] == loc]
    loc_missing = loc_df.isnull().sum().sum()
    print(f'   {loc:<25}: {loc_missing:>6,} NaN ({loc_missing / loc_df.size * 100:.3f}%)')

## 3️⃣ Thống Kê Mô Tả — Descriptive Statistics

In [ ]:
# Các cột chính cần phân tích
AQ_COLS = ['aqi', 'pm25', 'pm10', 'co', 'no2', 'so2', 'o3']
WEATHER_COLS = ['temp', 'rh', 'dewpt', 'wind_spd', 'precip', 'clouds']

# Chỉ lấy cột có trong DataFrame
aq_available = [c for c in AQ_COLS if c in df.columns]
weather_available = [c for c in WEATHER_COLS if c in df.columns]

print('📈 Thống kê mô tả — Chất lượng không khí:')
display(df[aq_available].describe().round(2))

print('\n🌤️  Thống kê mô tả — Thời tiết:')
display(df[weather_available].describe().round(2))

## 4️⃣ Phân Bố Biến AQ — Histograms + KDE

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(aq_available):
    ax = axes[i]
    data = df[col].dropna()
    
    ax.hist(data, bins=80, color=AQ_COLORS[i], alpha=0.7, edgecolor='white', density=True)
    data.plot.kde(ax=ax, color='black', linewidth=1.5)
    
    ax.set_title(f'{col.upper()}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    
    # Thêm thống kê
    textstr = f'μ={data.mean():.1f}\nσ={data.std():.1f}\nmed={data.median():.1f}'
    ax.text(0.95, 0.95, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Ẩn subplot thừa
for j in range(len(aq_available), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Phân Bố Các Chỉ Số Chất Lượng Không Khí', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('out/01_aq_distribution.png')
plt.show()

## 5️⃣ Phân Bố Biến Thời Tiết — Histograms

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

weather_colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12', '#7f8c8d']

for i, col in enumerate(weather_available):
    ax = axes[i]
    data = df[col].dropna()
    
    ax.hist(data, bins=60, color=weather_colors[i], alpha=0.7, edgecolor='white', density=True)
    data.plot.kde(ax=ax, color='black', linewidth=1.5)
    
    ax.set_title(f'{col}', fontweight='bold')
    textstr = f'μ={data.mean():.1f}\nσ={data.std():.1f}'
    ax.text(0.95, 0.95, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

for j in range(len(weather_available), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Phân Bố Các Biến Thời Tiết', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('out/02_weather_distribution.png')
plt.show()

## 6️⃣ Box Plot — PM2.5 theo Tỉnh/Thành

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Box plot PM2.5 theo tỉnh
province_order = df.groupby('province')['pm25'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='province', y='pm25', order=province_order,
            palette='RdYlGn_r', showfliers=False, ax=axes[0])
axes[0].set_title('PM2.5 theo Tỉnh/Thành', fontweight='bold')
axes[0].set_xlabel('Tỉnh/Thành')
axes[0].set_ylabel('PM2.5 (μg/m³)')
axes[0].tick_params(axis='x', rotation=45)

# Box plot PM2.5 theo trạm  
station_order = df.groupby('location')['pm25'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='location', y='pm25', order=station_order,
            palette='RdYlGn_r', showfliers=False, ax=axes[1])
axes[1].set_title('PM2.5 theo Trạm Đo', fontweight='bold')
axes[1].set_xlabel('Trạm')
axes[1].set_ylabel('PM2.5 (μg/m³)')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig('out/03_pm25_boxplot.png')
plt.show()

In [ ]:
# Box plot tất cả AQ columns
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for i, col in enumerate(aq_available):
    sns.boxplot(data=df, x='province', y=col, 
                palette='Set2', showfliers=False, ax=axes[i])
    axes[i].set_title(f'{col.upper()} theo Tỉnh', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_xlabel('')

for j in range(len(aq_available), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Phân Bố Các Chỉ Số AQ theo Tỉnh/Thành', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('out/04_aq_boxplot_province.png')
plt.show()

## 7️⃣ Correlation Heatmap — Ma Trận Tương Quan

In [ ]:
# Chọn các cột quan trọng cho heatmap
corr_cols = aq_available + weather_available
extra_cols = ['wind_sin', 'wind_cos', 'ah', 'dpd', 'is_stagnant',
              'rush_hour', 'pod', 'delta_pm25', 'ratio_pm']
corr_cols += [c for c in extra_cols if c in df.columns]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Ma Trận Tương Quan — AQ + Weather + Engineered Features', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('out/05_correlation_heatmap.png')
plt.show()

In [ ]:
# Top tương quan với PM2.5
if 'pm25' in df.columns:
    pm25_corr = corr_matrix['pm25'].drop('pm25').sort_values(key=abs, ascending=False)
    print('🔗 Top 15 biến tương quan mạnh nhất với PM2.5:')
    for feat, corr_val in pm25_corr.head(15).items():
        direction = '↑' if corr_val > 0 else '↓'
        print(f'   {direction} {feat:<20}: {corr_val:+.3f}')

## 8️⃣ Time Series — Chuỗi Thời Gian PM2.5

In [ ]:
# Chọn 4 trạm đại diện (Bắc, Trung, Nam)
sample_stations = df['location'].unique()[:4]

fig, axes = plt.subplots(len(sample_stations), 1, figsize=(20, 4*len(sample_stations)), sharex=True)
if len(sample_stations) == 1:
    axes = [axes]

for i, station in enumerate(sample_stations):
    station_df = df[df['location'] == station].sort_values('timestamp_local')
    
    axes[i].plot(station_df['timestamp_local'], station_df['pm25'], 
                 color=AQ_COLORS[i], alpha=0.6, linewidth=0.5)
    
    # Rolling average 24h
    if len(station_df) > 24:
        rolling_mean = station_df['pm25'].rolling(24).mean()
        axes[i].plot(station_df['timestamp_local'], rolling_mean,
                     color='black', linewidth=1.5, label='MA-24h')
    
    # Ngưỡng WHO
    axes[i].axhline(y=15, color='green', linestyle='--', alpha=0.5, label='WHO guideline (15 μg/m³)')
    axes[i].axhline(y=35, color='orange', linestyle='--', alpha=0.5, label='VN QCVN (35 μg/m³)')
    
    axes[i].set_ylabel('PM2.5 (μg/m³)')
    axes[i].set_title(f'📍 {station}', fontweight='bold', loc='left')
    axes[i].legend(loc='upper right', fontsize=9)
    axes[i].set_ylim(bottom=0)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

fig.suptitle('Chuỗi Thời Gian PM2.5 — Các Trạm Đại Diện', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('out/06_pm25_timeseries.png')
plt.show()

## 9️⃣ Mẫu Hình Theo Giờ — Hourly Patterns

In [ ]:
if 'hour' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # PM2.5 theo giờ — tất cả trạm
    hourly_pm25 = df.groupby('hour')['pm25'].agg(['mean', 'std'])
    axes[0,0].plot(hourly_pm25.index, hourly_pm25['mean'], 'o-', color='#e74c3c', linewidth=2)
    axes[0,0].fill_between(hourly_pm25.index, 
                           hourly_pm25['mean'] - hourly_pm25['std'],
                           hourly_pm25['mean'] + hourly_pm25['std'],
                           alpha=0.2, color='#e74c3c')
    axes[0,0].set_title('PM2.5 Trung Bình theo Giờ', fontweight='bold')
    axes[0,0].set_xlabel('Giờ trong ngày')
    axes[0,0].set_ylabel('PM2.5 (μg/m³)')
    axes[0,0].set_xticks(range(0, 24))
    # Highlight rush hours
    for rh in [(6,9), (16,19)]:
        axes[0,0].axvspan(rh[0], rh[1], alpha=0.1, color='orange', label='Rush hour' if rh[0]==6 else '')
    axes[0,0].legend()
    
    # PM2.5 theo giờ — theo tỉnh
    for province in df['province'].unique():
        prov_hourly = df[df['province'] == province].groupby('hour')['pm25'].mean()
        axes[0,1].plot(prov_hourly.index, prov_hourly.values, 'o-', label=province, markersize=3)
    axes[0,1].set_title('PM2.5 theo Giờ — Từng Tỉnh', fontweight='bold')
    axes[0,1].set_xlabel('Giờ')
    axes[0,1].set_ylabel('PM2.5 (μg/m³)')
    axes[0,1].legend(fontsize=8, ncol=3)
    axes[0,1].set_xticks(range(0, 24))
    
    # Nhiệt độ theo giờ
    hourly_temp = df.groupby('hour')['temp'].agg(['mean', 'std'])
    axes[1,0].plot(hourly_temp.index, hourly_temp['mean'], 'o-', color='#e67e22', linewidth=2)
    axes[1,0].fill_between(hourly_temp.index,
                           hourly_temp['mean'] - hourly_temp['std'],
                           hourly_temp['mean'] + hourly_temp['std'],
                           alpha=0.2, color='#e67e22')
    axes[1,0].set_title('Nhiệt Độ Trung Bình theo Giờ', fontweight='bold')
    axes[1,0].set_xlabel('Giờ')
    axes[1,0].set_ylabel('Temp (°C)')
    axes[1,0].set_xticks(range(0, 24))
    
    # Wind speed theo giờ
    hourly_wind = df.groupby('hour')['wind_spd'].agg(['mean', 'std'])
    axes[1,1].plot(hourly_wind.index, hourly_wind['mean'], 'o-', color='#3498db', linewidth=2)
    axes[1,1].fill_between(hourly_wind.index,
                           hourly_wind['mean'] - hourly_wind['std'],
                           hourly_wind['mean'] + hourly_wind['std'],
                           alpha=0.2, color='#3498db')
    axes[1,1].set_title('Tốc Độ Gió Trung Bình theo Giờ', fontweight='bold')
    axes[1,1].set_xlabel('Giờ')
    axes[1,1].set_ylabel('Wind Speed (m/s)')
    axes[1,1].set_xticks(range(0, 24))
    
    plt.tight_layout()
    plt.savefig('out/07_hourly_patterns.png')
    plt.show()

## 🔟 Mẫu Hình Theo Tháng — Monthly/Seasonal Patterns

In [ ]:
if 'month' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    month_names = ['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12']
    
    # PM2.5 theo tháng
    monthly_pm25 = df.groupby('month')['pm25'].agg(['mean','median','std'])
    x = monthly_pm25.index
    axes[0,0].bar(x, monthly_pm25['mean'], color='#e74c3c', alpha=0.7, label='Mean')
    axes[0,0].plot(x, monthly_pm25['median'], 'D-', color='black', label='Median')
    axes[0,0].set_title('PM2.5 Trung Bình theo Tháng', fontweight='bold')
    axes[0,0].set_xticks(range(1,13))
    axes[0,0].set_xticklabels(month_names)
    axes[0,0].legend()
    axes[0,0].set_ylabel('PM2.5 (μg/m³)')
    
    # AQI theo tháng
    monthly_aqi = df.groupby('month')['aqi'].agg(['mean','median'])
    axes[0,1].bar(monthly_aqi.index, monthly_aqi['mean'], color='#f39c12', alpha=0.7, label='Mean')
    axes[0,1].plot(monthly_aqi.index, monthly_aqi['median'], 'D-', color='black', label='Median')
    axes[0,1].set_title('AQI Trung Bình theo Tháng', fontweight='bold')
    axes[0,1].set_xticks(range(1,13))
    axes[0,1].set_xticklabels(month_names)
    axes[0,1].legend()
    axes[0,1].set_ylabel('AQI')
    
    # Temp theo tháng (heatmap style)
    for province in df['province'].unique():
        prov_monthly = df[df['province'] == province].groupby('month')['temp'].mean()
        axes[1,0].plot(prov_monthly.index, prov_monthly.values, 'o-', label=province, markersize=4)
    axes[1,0].set_title('Nhiệt Độ TB theo Tháng — Từng Tỉnh', fontweight='bold')
    axes[1,0].set_xticks(range(1,13))
    axes[1,0].set_xticklabels(month_names)
    axes[1,0].legend(fontsize=7, ncol=3)
    axes[1,0].set_ylabel('Temp (°C)')
    
    # PM2.5 theo tháng — Bắc vs Nam
    north = ['Hanoi', 'HaiPhong', 'ThanhHoa', 'NgheAn', 'NinhBinh']
    south = ['HCM', 'DongNai', 'CanTho', 'AnGiang', 'VinhLong']
    north_df = df[df['province'].isin(north)]
    south_df = df[df['province'].isin(south)]
    
    if len(north_df) > 0:
        north_monthly = north_df.groupby('month')['pm25'].mean()
        axes[1,1].plot(north_monthly.index, north_monthly.values, 'o-', 
                       color='#e74c3c', linewidth=2, label='Miền Bắc')
    if len(south_df) > 0:
        south_monthly = south_df.groupby('month')['pm25'].mean()
        axes[1,1].plot(south_monthly.index, south_monthly.values, 'o-',
                       color='#3498db', linewidth=2, label='Miền Nam')
    axes[1,1].set_title('PM2.5: Miền Bắc vs Miền Nam', fontweight='bold')
    axes[1,1].set_xticks(range(1,13))
    axes[1,1].set_xticklabels(month_names)
    axes[1,1].legend()
    axes[1,1].set_ylabel('PM2.5 (μg/m³)')
    
    plt.tight_layout()
    plt.savefig('out/08_monthly_patterns.png')
    plt.show()

## 1️⃣1️⃣ Quality Flags — Phân Tích Cờ Chất Lượng

In [ ]:
flag_cols = ['is_frozen', 'is_outlier']
flag_available = [c for c in flag_cols if c in df.columns]

if flag_available:
    fig, axes = plt.subplots(1, len(flag_available) + 1, figsize=(18, 6))
    
    for i, flag in enumerate(flag_available):
        counts = df[flag].value_counts()
        labels = ['Bình thường', 'Flagged']
        colors = ['#2ecc71', '#e74c3c']
        axes[i].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
                    startangle=90, textprops={'fontsize': 11})
        flag_label = 'Frozen Sensor' if flag == 'is_frozen' else 'Outlier'
        axes[i].set_title(f'{flag_label}\n(tổng: {int(counts.get(1, 0)):,})', fontweight='bold')
    
    # Quality flags theo trạm
    ax = axes[-1]
    flag_by_station = df.groupby('location')[flag_available].sum()
    flag_by_station.plot(kind='barh', ax=ax, color=['#3498db', '#e74c3c'])
    ax.set_title('Flags theo Trạm', fontweight='bold')
    ax.set_xlabel('Số lượng')
    ax.legend(['Frozen', 'Outlier'])
    
    plt.tight_layout()
    plt.savefig('out/09_quality_flags.png')
    plt.show()
    
    # Báo cáo chi tiết
    print('\n📋 Báo cáo Quality Flags:')
    for flag in flag_available:
        total = int(df[flag].sum())
        pct = total / len(df) * 100
        print(f'   {flag}: {total:,} ({pct:.2f}%)')
    
    print('\n   Theo trạm:')
    for loc in df['location'].unique():
        loc_df = df[df['location'] == loc]
        frozen = int(loc_df['is_frozen'].sum()) if 'is_frozen' in loc_df.columns else 0
        outlier = int(loc_df['is_outlier'].sum()) if 'is_outlier' in loc_df.columns else 0
        print(f'   {loc:<25}: frozen={frozen:>5,}  outlier={outlier:>5,}')
else:
    print('⚠️ Không tìm thấy cột quality flags (is_frozen, is_outlier)')

## 1️⃣2️⃣ Kiểm Tra Feature Engineering

In [ ]:
# Kiểm tra các engineered features
eng_features = {
    'Cyclic Time': ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'season_sin'],
    'Wind Components': ['wind_sin', 'wind_cos', 'spd_wind_sin', 'spd_wind_cos'],
    'Weather Derived': ['ah', 'dpd', 'is_stagnant', 'ratio_pm', 'o3_co'],
    'Time Context': ['hour', 'day', 'month', 'year', 'pod', 'rush_hour'],
    'Lag/Rolling': ['delta_pm25', 'ma_pm25_4', 'ma_pm25_24', 'rain_sum_6', 'w_pm25',
                    'pm25_lag_1', 'pm25_lag_3', 'pm25_lag_6', 'pm25_lag_24'],
    'Location': ['province', 'district', 'location']
}

print('🔧 Kiểm tra Feature Engineering:')
for group, features in eng_features.items():
    present = [f for f in features if f in df.columns]
    missing = [f for f in features if f not in df.columns]
    status = '✅' if not missing else '⚠️'
    print(f'\n   {status} {group}:')
    for f in present:
        if df[f].dtype in ['float64', 'int64']:
            print(f'      ✓ {f:<18} range=[{df[f].min():.2f}, {df[f].max():.2f}]  mean={df[f].mean():.2f}')
        else:
            print(f'      ✓ {f:<18} unique={df[f].nunique()}')
    for f in missing:
        print(f'      ✗ {f:<18} KHÔNG TÌM THẤY')

In [ ]:
# Scatter: Cyclic encoding kiểm tra
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if 'hour_sin' in df.columns and 'hour_cos' in df.columns:
    sample = df.sample(min(5000, len(df)))
    scatter = axes[0].scatter(sample['hour_cos'], sample['hour_sin'], 
                              c=sample['hour'], cmap='hsv', s=5, alpha=0.5)
    plt.colorbar(scatter, ax=axes[0], label='Hour')
    axes[0].set_title('Hour Cyclic Encoding (sin vs cos)', fontweight='bold')
    axes[0].set_xlabel('hour_cos')
    axes[0].set_ylabel('hour_sin')
    axes[0].set_aspect('equal')

if 'month_sin' in df.columns and 'month_cos' in df.columns:
    scatter2 = axes[1].scatter(sample['month_cos'], sample['month_sin'],
                               c=sample['month'], cmap='hsv', s=5, alpha=0.5)
    plt.colorbar(scatter2, ax=axes[1], label='Month')
    axes[1].set_title('Month Cyclic Encoding (sin vs cos)', fontweight='bold')
    axes[1].set_xlabel('month_cos')
    axes[1].set_ylabel('month_sin')
    axes[1].set_aspect('equal')

plt.tight_layout()
plt.savefig('out/10_cyclic_encoding_check.png')
plt.show()

## 1️⃣3️⃣ Kiểm Tra Tính Hợp Lệ Sau Imputation

In [ ]:
print('🔍 KIỂM TRA TÍNH HỢP LỆ DỮ LIỆU SAU TIỀN XỬ LÝ:')
print('=' * 60)

# 1. Giá trị âm
non_neg_cols = ['aqi', 'pm25', 'pm10', 'co', 'no2', 'so2', 'o3', 'wind_spd', 'precip', 'rh']
non_neg_available = [c for c in non_neg_cols if c in df.columns]
print('\n1. Kiểm tra giá trị âm (không hợp lệ):')
has_issue = False
for col in non_neg_available:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f'   ❌ {col}: {neg_count:,} giá trị âm!')
        has_issue = True
if not has_issue:
    print('   ✅ Tất cả cột non-negative đều >= 0')

# 2. Giá trị extreme
print('\n2. Kiểm tra giá trị extreme:')
THRESHOLDS_CHECK = {
    'pm25': 600, 'pm10': 800, 'aqi': 500,
    'temp': 50, 'rh': 105, 'wind_spd': 50
}
for col, max_val in THRESHOLDS_CHECK.items():
    if col in df.columns:
        extreme = (df[col] > max_val).sum()
        actual_max = df[col].max()
        status = '✅' if extreme == 0 else '⚠️'
        print(f'   {status} {col:<10}: max={actual_max:.1f} (threshold={max_val}), extreme={extreme}')

# 3. ratio_pm check
if 'ratio_pm' in df.columns:
    print(f'\n3. ratio_pm: max={df["ratio_pm"].max():.2f}, '
          f'mean={df["ratio_pm"].mean():.2f}, '
          f'>10: {(df["ratio_pm"] > 10).sum()}')

# 4. dpd check  
if 'dpd' in df.columns:
    print(f'\n4. dpd (Dew Point Depression): '
          f'min={df["dpd"].min():.1f}, max={df["dpd"].max():.1f}, '
          f'mean={df["dpd"].mean():.1f}')
    neg_dpd = (df['dpd'] < 0).sum()
    print(f'   DPD < 0 (supersaturation): {neg_dpd:,} ({neg_dpd/len(df)*100:.2f}%)')

# 5. Lag features check
print('\n5. Lag features — kiểm tra fillna(0) đã được fix:')
lag_cols = ['pm25_lag_1', 'pm25_lag_3', 'pm25_lag_6', 'pm25_lag_24']
for col in lag_cols:
    if col in df.columns:
        zero_count = (df[col] == 0).sum()
        pct = zero_count / len(df) * 100
        status = '✅' if pct < 1 else '⚠️'
        print(f'   {status} {col}: zeros={zero_count:,} ({pct:.2f}%)')

print('\n' + '=' * 60)
print('✅ KIỂM TRA HOÀN TẤT')

## 1️⃣4️⃣ Bảng Tổng Kết cho Báo Cáo

In [ ]:
# Tạo bảng tổng kết có thể copy vào báo cáo
print('📝 BẢNG TỔNG KẾT DỮ LIỆU — Dùng cho Báo Cáo/Luận Văn')
print('=' * 80)

summary_data = []
all_check_cols = aq_available + weather_available

for col in all_check_cols:
    summary_data.append({
        'Biến': col,
        'Count': f'{df[col].count():,}',
        'Mean': f'{df[col].mean():.2f}',
        'Std': f'{df[col].std():.2f}',
        'Min': f'{df[col].min():.2f}',
        'Q1': f'{df[col].quantile(0.25):.2f}',
        'Median': f'{df[col].median():.2f}',
        'Q3': f'{df[col].quantile(0.75):.2f}',
        'Max': f'{df[col].max():.2f}',
        'Skew': f'{df[col].skew():.2f}',
        'Kurt': f'{df[col].kurtosis():.2f}',
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

# Lưu ra CSV
summary_df.to_csv('out/summary_statistics.csv', index=False)
print('\n💾 Đã lưu: out/summary_statistics.csv')

In [ ]:
# Bảng thống kê theo trạm
print('\n📊 PM2.5 Statistics theo Trạm:')
station_stats = df.groupby(['province', 'location'])['pm25'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(2)
station_stats.columns = ['Số mẫu', 'Mean', 'Std', 'Min', 'Median', 'Max']
display(station_stats)

station_stats.to_csv('out/pm25_station_statistics.csv')
print('\n💾 Đã lưu: out/pm25_station_statistics.csv')

In [ ]:
# Thông tin tổng kết cho phần Methodology của luận văn
print('\n' + '='*80)
print('📄 THÔNG TIN CHO PHẦN METHODOLOGY CỦA BÁO CÁO')
print('='*80)

total_hours = len(df)
n_stations = df['location'].nunique()
n_provinces = df['province'].nunique()
date_range = f"{df['timestamp_local'].min().strftime('%Y-%m-%d')} đến {df['timestamp_local'].max().strftime('%Y-%m-%d')}"
n_features = len([c for c in df.columns if c not in ['timestamp_local', 'province', 'district', 'location']])

frozen_pct = df['is_frozen'].mean() * 100 if 'is_frozen' in df.columns else 0
outlier_pct = df['is_outlier'].mean() * 100 if 'is_outlier' in df.columns else 0

print(f'''
Dữ liệu nghiên cứu:
  - Quy mô: {total_hours:,} bản ghi hourly từ {n_stations} trạm quan trắc
    thuộc {n_provinces} tỉnh/thành phố tại Việt Nam
  - Giai đoạn: {date_range}
  - Số features: {n_features} (bao gồm raw + engineered)
  
Tiền xử lý:
  - Missing values: Two-stage imputation
    (1) Linear interpolation cho gap ≤6h
    (2) KNN Imputation (k=12, StandardScaler) cho gap lớn
  - Frozen sensor: {frozen_pct:.2f}% dữ liệu bị flag (rolling 12h, std < 1e-6)
  - Outlier: {outlier_pct:.2f}% (physical thresholds + IQR 3×)
  - Feature engineering: cyclic time encoding, wind components,
    weather derivatives (AH, DPD), lag features (1/3/6/24h),
    rolling means (4h/24h)
''')

In [ ]:
print('\n🎉 PHÂN TÍCH HOÀN TẤT!')
print('\nCác file đã tạo trong thư mục out/:')
import os
if os.path.exists('out'):
    for f in sorted(os.listdir('out')):
        size = os.path.getsize(os.path.join('out', f)) / 1024
        print(f'   📄 {f} ({size:.1f} KB)')